In [3]:
# ===============================
# 1. Imports
# ===============================
import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
from scipy.signal import butter, filtfilt
import json
import os

os.chdir('/Users/aryantripathi/P300-EEG-Authentication')

# ===============================
# 2. Load A01.mat correctly
# ===============================
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

# Extract fields
X = s1['X']  # Shape: (347704, 8)
y_stim = s1['y_stim']  # Shape: (347704, 1) - stimulus labels

print("X shape:", X.shape)
print("y_stim shape:", y_stim.shape)

# ===============================
# 3. Convert to (channels, samples)
# ===============================
eeg = X.T  # Transpose to (8, 347704)
target = y_stim.flatten()  # Flatten to (347704,)

print("EEG shape (channels, samples):", eeg.shape)
print("Target shape:", target.shape)
print("Unique target values:", np.unique(target))

# ===============================
# 4. Bandpass filter
# ===============================
fs = 240

def bandpass_filter(signal, lowcut=0.1, highcut=30, fs=240, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal, padlen=0)

filtered_eeg = np.zeros_like(eeg, dtype=float)
for ch in range(eeg.shape[0]):
    filtered_eeg[ch] = bandpass_filter(eeg[ch].astype(float))

print("Filtered EEG shape:", filtered_eeg.shape)

# ===============================
# 5. Plot 1 — RAW EEG
# ===============================
eeg_channel_index = 2
plt.figure(figsize=(10,3))
plt.plot(eeg[eeg_channel_index, :2000].astype(float))
plt.title("Raw EEG Signal (Subject A01, Channel 2)")
plt.xlabel("Samples")
plt.ylabel("Amplitude")
plt.grid(True)
plt.savefig("plots/raw_eeg_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 6. Plot 2 — RAW vs FILTERED EEG
# ===============================
plt.figure(figsize=(10,6))

plt.subplot(2,1,1)
plt.plot(eeg[eeg_channel_index, :2000].astype(float))
plt.title("Raw EEG")

plt.subplot(2,1,2)
plt.plot(filtered_eeg[eeg_channel_index, :2000])
plt.title("Filtered EEG (0.1–30 Hz)")

plt.tight_layout()
plt.savefig("plots/raw_vs_filtered_eeg_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 7. Epoch extraction (-200 ms to +800 ms)
# ===============================
baseline_samples = int(0.2 * fs)
post_samples = int(0.8 * fs)

# Find target stimulus indices (where target == 1)
target_indices = np.where(target == 1)[0]
print(f"Found {len(target_indices)} target events")

epochs = []
for idx in target_indices:
    start = idx - baseline_samples
    end = idx + post_samples
    if start >= 0 and end < filtered_eeg.shape[1]:
        epoch = filtered_eeg[:, start:end]
        epochs.append(epoch)

epochs = np.array(epochs)
print("Epochs shape (trials, channels, samples):", epochs.shape)

# ===============================
# 8. Baseline correction + averaging
# ===============================
baseline_corrected_epochs = np.zeros_like(epochs, dtype=float)

for trial_idx in range(epochs.shape[0]):
    for ch in range(epochs.shape[1]):
        epoch = epochs[trial_idx, ch, :]
        baseline = epoch[:baseline_samples].mean()
        baseline_corrected_epochs[trial_idx, ch, :] = epoch - baseline

# Average across trials
p300_template = baseline_corrected_epochs.mean(axis=0)
time = np.linspace(-0.2, 0.8, p300_template.shape[1])

print("P300 template shape (channels, samples):", p300_template.shape)

# ===============================
# 9. Plot 3 — FINAL AVERAGED P300
# ===============================
plt.figure(figsize=(10,4))
plt.plot(time, p300_template[eeg_channel_index, :])
plt.axvline(0, color='r', linestyle='--', label='Stimulus')
plt.title("Averaged P300 Waveform (Subject A01, Channel 2)")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.savefig("plots/averaged_p300_A01_channel2.png", dpi=300)
plt.close()

# ===============================
# 10. SAVE PREPROCESSED DATA
# ===============================
os.makedirs('data/preprocessed', exist_ok=True)

# Save epochs
np.save('data/preprocessed/A01_epochs.npy', baseline_corrected_epochs)

# Save metadata
metadata = {
    'subject': 'A01',
    'n_epochs': len(baseline_corrected_epochs),
    'n_channels': baseline_corrected_epochs.shape[1],
    'n_samples': baseline_corrected_epochs.shape[2],
    'sampling_rate': fs,
    'channels': ['Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8'],
    'baseline_duration_ms': 200,
    'post_stimulus_duration_ms': 800
}

with open('data/preprocessed/A01_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("✓ Saved A01_epochs.npy")
print("✓ Saved A01_metadata.json")
print(f"✓ Total epochs saved: {baseline_corrected_epochs.shape[0]}")

X shape: (347704, 8)
y_stim shape: (347704, 1)
EEG shape (channels, samples): (8, 347704)
Target shape: (347704,)
Unique target values: [ 0  1  2  3  4  5  6  7  8  9 10 11 12]
Filtered EEG shape: (8, 347704)
Found 11200 target events
Epochs shape (trials, channels, samples): (11200, 8, 240)
P300 template shape (channels, samples): (8, 240)
✓ Saved A01_epochs.npy
✓ Saved A01_metadata.json
✓ Total epochs saved: 11200


In [9]:
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import json
import os

os.chdir('/Users/aryantripathi/Desktop/P300-EEG-Authentication')

# ===============================
# Process ALL healthy subjects A01S-A10S
# ===============================
subjects = ['A01S', 'A02S', 'A03S', 'A04S', 'A05S', 'A06S', 'A07S', 'A08S', 'A09S', 'A10S']
fs = 256

def bandpass_filter(signal, lowcut=0.1, highcut=30, fs=256, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal, padlen=0)

for subject in subjects:
    print(f"\n{'='*60}")
    print(f"Processing {subject}")
    print(f"{'='*60}")
    
    try:
        # Load data
        data = sio.loadmat(f'data/raw/{subject}.mat')
        s1 = data['data'][0][0]
        
        # Extract X and y - handle nested structure
        X_field = s1['X']
        y_stim_field = s1['y_stim']
        
        # Extract from nested structure
        if X_field.shape == (1, 1):
            X = X_field[0][0]
        else:
            X = X_field
            
        if y_stim_field.shape == (1, 1):
            y_stim = y_stim_field[0][0]
        else:
            y_stim = y_stim_field
        
        print(f"X shape: {X.shape}")
        print(f"y_stim shape: {y_stim.shape}")
        
        # Convert to (channels, samples)
        eeg = X.T.astype(float)
        target = y_stim.flatten().astype(int)
        
        # Bandpass filter
        filtered_eeg = np.zeros_like(eeg, dtype=float)
        for ch in range(eeg.shape[0]):
            filtered_eeg[ch] = bandpass_filter(eeg[ch], fs=fs)
        
        # Epoch extraction
        baseline_samples = int(0.1 * fs)  # 100ms baseline for healthy subjects
        post_samples = int(0.8 * fs)     # 800ms post-stimulus
        
        target_indices = np.where(target == 1)[0]
        print(f"Found {len(target_indices)} target events")
        
        epochs = []
        for idx in target_indices:
            start = idx - baseline_samples
            end = idx + post_samples
            if start >= 0 and end < filtered_eeg.shape[1]:
                epoch = filtered_eeg[:, start:end]
                epochs.append(epoch)
        
        epochs = np.array(epochs)
        print(f"Epochs shape: {epochs.shape}")
        
        # Baseline correction
        baseline_corrected_epochs = np.zeros_like(epochs, dtype=float)
        
        for trial_idx in range(epochs.shape[0]):
            for ch in range(epochs.shape[1]):
                epoch = epochs[trial_idx, ch, :]
                baseline = epoch[:baseline_samples].mean()
                baseline_corrected_epochs[trial_idx, ch, :] = epoch - baseline
        
        # Save
        os.makedirs('data/preprocessed', exist_ok=True)
        np.save(f'data/preprocessed/{subject}_epochs.npy', baseline_corrected_epochs)
        
        # Save metadata
        metadata = {
            'subject': subject,
            'dataset': 'BNCI 009-2014 (Healthy Subjects)',
            'n_epochs': len(baseline_corrected_epochs),
            'n_channels': baseline_corrected_epochs.shape[1],
            'n_samples': baseline_corrected_epochs.shape[2],
            'sampling_rate': fs,
            'baseline_duration_ms': 100,
            'post_stimulus_duration_ms': 800
        }
        
        with open(f'data/preprocessed/{subject}_metadata.json', 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"✓ {subject}: {baseline_corrected_epochs.shape[0]} epochs saved")
        
    except Exception as e:
        print(f"✗ {subject}: Error - {str(e)}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print("✅ All healthy subjects processed!")
print(f"{'='*60}")


Processing A01S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A01S: 1536 epochs saved

Processing A02S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A02S: 1536 epochs saved

Processing A03S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A03S: 1536 epochs saved

Processing A04S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A04S: 1536 epochs saved

Processing A05S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A05S: 1536 epochs saved

Processing A06S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A06S: 1536 epochs saved

Processing A07S
X shape: (50184, 16)
y_stim shape: (50184, 1)
Found 1536 target events
Epochs shape: (1536, 16, 229)
✓ A07S: 1536 epoch

In [11]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

y_field = s1['y']
print("y_field shape:", y_field.shape)
print("y_field type:", type(y_field))

y_stim = s1['y_stim']
print("y_stim shape:", y_stim.shape)
print("y_stim[0][0] shape:", y_stim[0][0].shape if y_stim.shape == (1,1) else "not nested")
print("y_stim content sample:", y_stim[0][0][:20] if hasattr(y_stim[0][0], '__getitem__') else y_stim[0][0])

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/A01.mat'

In [7]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data'][0][0]

# Check what's in X
X_field = s1['X']
print("X_field shape:", X_field.shape)ell5:ell5:
print("X_field type:", type(X_field))
print("X_field:", X_field)

if X_field.shape == (1, 1):
    X_data = X_field[0][0]
    print("X_data shape:", X_data.shape)
    print("X_data type:", type(X_data))

SyntaxError: invalid non-printable character U+0008 (4214570489.py, line 7)

In [9]:
import scipy.io as sio
import numpy as np

data = sio.loadmat('data/raw/A01.mat')
s1 = data['data']

print("s1 shape:", s1.shape)
print("s1 type:", type(s1))
print("s1[0][0] fields:", s1[0][0].dtype.names if hasattr(s1[0][0], 'dtype') else 'not a struct')

# Try to access X directly
try:
    X = s1['X']
    print("X shape:", X.shape)
except:
    try:
        X = s1[0][0]['X']
        print("X[0][0] shape:", X.shape)
    except:
        print("Cannot access X")

s1 shape: (1, 1)
s1 type: <class 'numpy.ndarray'>
s1[0][0] fields: ('channels', 'X', 'y', 'y_stim', 'trial', 'classes', 'classes_stim', 'gender', 'age', 'ALSfrs', 'onsetALS')
X shape: (1, 1)


In [5]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
print(data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'data'])


In [6]:
import scipy.io as sio
data = sio.loadmat('data/raw/A01.mat')
s1 = data['data']
print(s1.dtype)
print(s1.shape)

[('channels', 'O'), ('X', 'O'), ('y', 'O'), ('y_stim', 'O'), ('trial', 'O'), ('classes', 'O'), ('classes_stim', 'O'), ('gender', 'O'), ('age', 'O'), ('ALSfrs', 'O'), ('onsetALS', 'O')]
(1, 1)
